In [11]:
with open("input.txt",'r',encoding='utf-8') as f:
    text = f.read()

In [12]:
print(f"Total number of characters = {len(text)}")

Total number of characters = 1115394


In [13]:
print(text[:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [14]:
chars = sorted(list(set(text)))
num_of_unique_chars = len(chars)
print(" ".join(chars))
print(num_of_unique_chars)
vocab_size = num_of_unique_chars


   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z
65


In [15]:
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for i,s in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda c: "".join([itos[s] for s in c])

print(encode("bruh less go"))
print(decode(encode("bruh less go")))

[40, 56, 59, 46, 1, 50, 43, 57, 57, 1, 45, 53]
bruh less go


In [16]:
import torch

data = torch.tensor(encode(text),dtype =  torch.long)

n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [17]:
context_length = 8
x = train_data[:context_length]
y = train_data[1:context_length+1]
for i in range(context_length):
    context = x[:i+1]
    target = y[i]
    print(f"For the context of {context} the target would be {target}")

For the context of tensor([18]) the target would be 47
For the context of tensor([18, 47]) the target would be 56
For the context of tensor([18, 47, 56]) the target would be 57
For the context of tensor([18, 47, 56, 57]) the target would be 58
For the context of tensor([18, 47, 56, 57, 58]) the target would be 1
For the context of tensor([18, 47, 56, 57, 58,  1]) the target would be 15
For the context of tensor([18, 47, 56, 57, 58,  1, 15]) the target would be 47
For the context of tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target would be 58


In [18]:
torch.manual_seed(1337)
batch_size = 4
def get_batch(split):
    data = {"training":train_data,
     "validation":val_data}[split]
    ix = torch.randint(len(data)-context_length,(batch_size,))
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:context_length+i+1] for i in ix])
    return x,y 

xb,yb = get_batch("training")
print('inputs:')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)

print('---')

for b in range(batch_size):
    for c in range(context_length):
        context = xb[b,:c+1]
        target = yb[b,c]
        print(f'When the context is {context} the target is {target}')



inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
---
When the context is tensor([24]) the target is 43
When the context is tensor([24, 43]) the target is 58
When the context is tensor([24, 43, 58]) the target is 5
When the context is tensor([24, 43, 58,  5]) the target is 57
When the context is tensor([24, 43, 58,  5, 57]) the target is 1
When the context is tensor([24, 43, 58,  5, 57,  1]) the target is 46
When the context is tensor([24, 43, 58,  5, 57,  1, 46]) the target is 43
When the context is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target is 39
When the context is tensor([44]) the target is 53
When the context is tensor([44, 53]) the tar

In [19]:
from torch.nn import functional as F
import torch.nn as nn

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
    
    def forward(self,idx,targets = None):
        logits = self.token_embedding_table(idx) #(B,T,C)
        if targets == None:
            loss = None
        else:
            B,T,C = logits.shape
            logitst = logits.view(B*T,C) # Reshaping the different batches to be of one and making sure C (num of classes) is the second shape of logits
            targets = targets.view(B*T) # Since the targets should match the logits first shape 
            loss = F.cross_entropy(logitst,targets) 
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        #idx has shape B,T (say 4, 8)
        for _ in range(max_new_tokens):
            logits,loss = self(idx) # (B,T,C) in this case 4,8,65 for now 

            logits = logits[:,-1,:] # Taking the logits of the last character in every batch so B,C 4,65

            probs = F.softmax(logits,dim =1)

            idx_next = torch.multinomial(probs,num_samples=1) # B,1 - it predicts what the next character is for evert batch 4,1

            idx = torch.cat((idx,idx_next),dim=1) # add the next coming element to our new idx which would be B,T+1 4,9
        return idx 
m = BigramLanguageModel(vocab_size)
logits,loss = m(xb,yb) # input as (batch_size,time(context_length)) , output as batch_size,time,vocab_size

print(decode(m.generate(idx = torch.zeros((1,1) , dtype= torch.long),max_new_tokens=100)[0].tolist()))





SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [20]:
optimizer = torch.optim.AdamW(m.parameters(),lr= 1e-3)

In [21]:
batch_size = 32
for steps in range(10000):
    xb,yb = get_batch("training")
    logits,loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

2.382369041442871


In [22]:
print(decode(m.generate(idx = torch.zeros((1,1) , dtype= torch.long),max_new_tokens=400)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer,


In [23]:
torch.manual_seed(1337)
B,T,C = 4,8,2 #batch,time,channels
x = torch.randn(B,T,C)
x.shape


torch.Size([4, 8, 2])

In [25]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev,0)

In [28]:
torch.manual_seed(42)
'''a = torch.tril(torch.ones(3,3))
a = a / a.sum(dim = 1,keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b 
print(a)
print(b)
print(c)'''
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(dim = 1, keepdim = True)
xbow2 = wei @ x # T,T @ B,T,C  , SINCE they dont match torch does this - B , T , T @ B , T , C giving a B , T , C 
print(xbow2[0])

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [29]:
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)
# self attention head stuff
head_size = 16
key = nn.Linear(C,head_size,bias = False) 
query = nn.Linear(C,head_size,bias= False)
value = nn.Linear(C,head_size,bias= False)
k = key(x) # B , T, 16
q = query(x) # B , T, 16
wei = q @ k.transpose(-2,-1) #B , T, T

tril = torch.tril(torch.ones(T,T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0,float('-inf'))
wei = F.softmax(wei,dim=-1) # B,T,T
v = value(x)  #B,T,T
#out = wei @ x
out = wei @ v # B,T,head_size
print(out[0])

tensor([[ 0.1808, -0.0700, -0.3596, -0.9152,  0.6258,  0.0255,  0.9545,  0.0643,
          0.3612,  1.1679, -1.3499, -0.5102,  0.2360, -0.2398, -0.9211,  1.5433,
          1.3488, -0.1396,  0.2858,  0.9651, -2.0371,  0.4931,  1.4870,  0.5910,
          0.1260, -1.5627, -1.1601, -0.3348,  0.4478, -0.8016,  1.5236,  2.5086],
        [-0.5303, -0.2227,  0.7946, -0.0416,  0.2320,  0.9596, -0.8221, -0.2413,
         -0.3708, -0.5947,  0.2482, -1.3398, -0.9788,  0.4441, -0.6483, -0.3416,
          1.5988, -0.6986,  1.1837, -0.0806, -1.5937,  1.8511,  2.5621, -1.3786,
          1.2430, -1.5185,  0.5093, -0.2309,  0.7268,  1.1658,  1.5962,  0.0550],
        [-0.5943,  0.3186,  0.0590, -0.2116, -0.1547,  0.4838, -0.1518, -0.7044,
          1.2507, -0.2447, -0.0523,  0.0842, -1.0431,  0.6117, -0.2001,  0.3510,
          2.1127, -0.9281,  0.9154,  0.5045, -1.6725, -0.3468,  1.1978, -0.2869,
          0.4814, -0.7891,  0.1911, -0.5849, -0.0408, -0.1889, -0.0443,  0.2009],
        [-0.3202, -0.1119

In [ ]:
class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
  
  def __call__(self, x):
    # calculate the forward pass 
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]